In [4]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
import numpy as np
from safetensors.torch import load_file

# Add parent directory to path to import utils if needed
sys.path.append(str(Path.cwd().parent))

# Configuration
RESULTS_DIR = Path("../results/cocoqa_img_vs_txt/similarities")
# Base path for ID measures (results directory)
ID_RESULTS_BASE = Path("../results")
PLOTS_DIR = Path("../plots/multimodality_similarity")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "Pretrained": "llava-v1.5-7b-pretrained-lbasile",
    "Finetuned": "llava-1.5-7b-hf"
}

# Mapping for ID data folder names (different from similarity filenames)
ID_MODEL_NAMES = {
    "Pretrained": "llava-v1.5-mlp2x-336px-pretrain-vicuna-7b-v1.5",
    "Finetuned": "llava-1.5-7b-hf"
}

MEASURES = {
    "neighborhood_overlap": "Neighborhood Overlap",
    "linear_cka": "Linear CKA",
    "distance_correlation": "Distance Correlation"
}

In [5]:
def load_data(model_name, measure_key):
    filename = f"{model_name}_{measure_key}.safetensors"
    path = RESULTS_DIR / filename
    try:
        data = load_file(str(path))
        # The key inside safetensors matches the measure name (e.g. "neighborhood_overlap")
        tensor = data[measure_key]
        return tensor.cpu().numpy()
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return None

def load_id_data(dataset_key, model_key):
    """
    Load Intrinsic Dimension data.
    dataset_key: 'cocoqa_img' or 'cocoqa_txt'
    model_key: 'Pretrained' or 'Finetuned'
    """
    model_dir = ID_MODEL_NAMES[model_key]
    # Verify path construction
    # e.g. ../results/cocoqa_img/models_data_measures/llava-1.5-7b-hf/id_16_100_output_layer_last.safetensors
    path = ID_RESULTS_BASE / dataset_key / "models_data_measures" / model_dir / "id_16_100_output_layer_last.safetensors"
    
    try:
        data = load_file(str(path))
        tensor = data["layers_intrinsic_dimension"]
        return tensor.cpu().numpy()
    except Exception as e:
        print(f"Error loading ID data from {path}: {e}")
        return None

def plot_measure_comparison(measure_key, measure_title):
    # Load data for both models (Similarity + ID)
    sim_pm = load_data(MODELS["Pretrained"], measure_key)
    sim_fm = load_data(MODELS["Finetuned"], measure_key)
    
    # ID Data: Image (Top/X) corresponds to cocoqa_img
    id_img_pm = load_id_data("cocoqa_img", "Pretrained")
    id_img_fm = load_id_data("cocoqa_img", "Finetuned")
    
    # ID Data: Text (Right/Y) corresponds to cocoqa_txt
    id_txt_pm = load_id_data("cocoqa_txt", "Pretrained")
    id_txt_fm = load_id_data("cocoqa_txt", "Finetuneds")

    if any(x is None for x in [sim_pm, sim_fm, id_img_pm, id_txt_pm, id_img_fm, id_txt_fm]):
        print(f"Skipping plot for {measure_title} due to missing data.")
        return

    # Calculate differences (Pretrained - Finetuned)
    sim_diff = sim_pm - sim_fm
    id_img_diff = id_img_pm - id_img_fm
    id_txt_diff = id_txt_pm - id_txt_fm
    
    # Determine scale for difference heatmap (centered at 0)
    max_diff = max(abs(np.nanmin(sim_diff)), abs(np.nanmax(sim_diff)))

    # Setup figure
    # We now have 3 columns: Pretrained, Finetuned, Difference
    fig = plt.figure(figsize=(26, 10))
    
    # GridSpec: 2 Rows (Plots, Colorbars), 3 Columns
    # Height ratios: Plots vs Colorbar height
    # Width ratios: Equal width for all 3 plots
    # hspace: Space between plots and colorbar
    # wspace: Space between columns
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 0.08], width_ratios=[1, 1, 1], hspace=0.25, wspace=0.2)
    
    # Configuration for each subplot
    models_config = [
        # (Label, SimMatrix, ID_Img, ID_Txt, Colormap, Vmin, Vmax)
        ("Pretrained", sim_pm, id_img_pm, id_txt_pm, "viridis", 0, 1),
        ("Finetuned", sim_fm, id_img_fm, id_txt_fm, "viridis", 0, 1),
        ("Difference (PM - FM)", sim_diff, id_img_diff, id_txt_diff, "coolwarm", -max_diff, max_diff)
    ]
    
    main_heatmaps = [] # To store mappables for colorbars
    
    for idx, (model_label, sim_data, id_img, id_txt, cmap, vmin, vmax) in enumerate(models_config):
        # Create Inner Grid for each model: 2x2 for margins
        # gs[0, idx] targets the plot area for the current model
        gs_inner = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[0, idx],
                                                    width_ratios=[6, 1], height_ratios=[1, 6],
                                                    wspace=0.03, hspace=0.03)
        
        ax_top = fig.add_subplot(gs_inner[0, 0])
        ax_main = fig.add_subplot(gs_inner[1, 0])
        ax_right = fig.add_subplot(gs_inner[1, 1])
        
        # Share axes for alignment
        ax_top.sharex(ax_main)
        ax_right.sharey(ax_main)
        
        # 1. Main Heatmap
        hm = sns.heatmap(
            sim_data,
            ax=ax_main,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            center=0 if idx == 2 else None, # Center difference plot
            square=False,
            cbar=False
        )
        # Store mappable if it's the first of its group (Pretrained for shared, Diff for diff)
        if idx == 0 or idx == 2:
            main_heatmaps.append(hm.collections[0])
        
        # Centers for 1D plots
        x_centers = np.arange(len(id_img)) + 0.5
        y_centers = np.arange(len(id_txt)) + 0.5
        
        # 2. Top Plot (Image ID)
        ax_top.plot(x_centers, id_img, color='#377eb8', lw=2) # Blue-ish
        # fill_between with 0 baseline handles both positive (regular ID) and negative (Difference)
        ax_top.fill_between(x_centers, id_img, 0, color='#377eb8', alpha=0.3)
        
        # Display ID axis (Y-axis) and labels
        sns.despine(ax=ax_top, bottom=True, left=False)
        ax_top.tick_params(bottom=False, labelbottom=False)
        ax_top.tick_params(axis='y', labelsize=8)
        if idx == 0:
            ax_top.set_ylabel("ID", fontsize=9, rotation=0, labelpad=15)
        elif idx == 2:
            ax_top.set_ylabel("Δ ID", fontsize=9, rotation=0, labelpad=15)
        
        # 3. Right Plot (Text ID)
        ax_right.plot(id_txt, y_centers, color='#e41a1c', lw=2) # Red-ish
        # fill_betweenx with 0 baseline
        ax_right.fill_betweenx(y_centers, id_txt, 0, color='#e41a1c', alpha=0.3)
        
        # Display ID axis (X-axis) and labels
        sns.despine(ax=ax_right, left=True, bottom=False)
        ax_right.tick_params(left=False, labelleft=False)
        ax_right.tick_params(axis='x', labelsize=8)
        if idx == 0:
            ax_right.set_xlabel("ID", fontsize=9)
        elif idx == 2:
            ax_right.set_xlabel("Δ ID", fontsize=9)
        
        # Formatting Main Axis
        ax_main.set_xlabel("Image Layers", fontsize=13, labelpad=8)
        ax_main.set_ylabel("Text Layers", fontsize=13, labelpad=8)
        ax_main.invert_yaxis() # 0 at bottom
        
        # Ticks
        num_layers = sim_data.shape[0]
        ticks = range(0, num_layers, 5)
        ax_main.set_xticks([t + 0.5 for t in ticks])
        ax_main.set_xticklabels(ticks, fontsize=11)
        ax_main.set_yticks([t + 0.5 for t in ticks])
        ax_main.set_yticklabels(ticks, fontsize=11)
        
        # Add model label using title on top ax
        ax_top.set_title(f"{model_label}", fontsize=18, pad=10, fontweight='medium')

    # Colorbars
    
    # 1. Shared Colorbar for Pretrained and Finetuned (Columns 0 and 1)
    # We span the first two columns of the bottom row
    cax_shared = fig.add_subplot(gs[1, 0:2])
    # Adjust width of colorbar within the slot if needed, or let it fill
    plt.colorbar(main_heatmaps[0], cax=cax_shared, orientation='horizontal', label=f"{measure_title} (0 to 1)")
    
    # 2. Separate Colorbar for Difference (Column 2)
    cax_diff = fig.add_subplot(gs[1, 2])
    plt.colorbar(main_heatmaps[1], cax=cax_diff, orientation='horizontal', label=f"Difference ({measure_title})")
    
    # Adjust layout
    fig.suptitle(f"{measure_title}: Text vs Image Modality Similarity & differences",
                 fontsize=22, fontweight='bold', y=0.98)
    
    # Save
    save_path = PLOTS_DIR / f"multimodality_{measure_key}_comparison_with_id.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor='white')
    print(f"Saved plot to {save_path}")
    plt.show()

In [6]:
# Generate plots for all measures
for key, title in MEASURES.items():
    plot_measure_comparison(key, title)

KeyError: 'Finetuneds'